# Staff/customer YOLOv8 training on Kaggle

This notebook trains a two-class detector on the supplied Roboflow YOLOv8 export: `customer` and `staff`. Attach the uploaded dataset as a Kaggle Dataset, turn on a GPU accelerator, then run all cells.

> **Important limitation:** the supplied data contains neighbouring video frames in different train, validation, and test splits. The resulting validation/test metrics are useful only as a smoke test for this export; they are not evidence of generalization to another camera, store, or uniform. Do not use the model to assign authoritative employee/customer roles.

## Kaggle setup

1. Create a new Kaggle Notebook and select **GPU T4 x2** (or another GPU) in *Notebook options*.
2. Add the `staff_customer.v2i.yolov8` folder as an input dataset. Keep its `train`, `valid`, `test`, and `data.yaml` structure intact.
3. Upload this notebook or paste the cells into the Kaggle editor.

The notebook finds the attached dataset automatically, so it does not depend on the Kaggle dataset slug.

In [ ]:
# Kaggle usually includes PyTorch. This installs/updates the YOLO training package.
!pip install -q -U ultralytics pyyaml

In [ ]:
from collections import Counter
from pathlib import Path
import platform
import random

import torch
import yaml
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training. CPU training will be very slow.')

DEVICE = 0

In [ ]:
# Exact Kaggle input layout shown in the attached file tree.
DATASET_ROOT = Path('/kaggle/input/staff-customer-comp-vision/staff_customer.v2i.yolov8')
required_paths = [
    DATASET_ROOT / 'data.yaml',
    *(DATASET_ROOT / split / kind for split in ('train', 'valid', 'test') for kind in ('images', 'labels')),
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Expected Kaggle dataset path was not found. Missing:\n' + '\n'.join(missing_paths) +
        '\nIf Kaggle assigned a different input slug, update only DATASET_ROOT in this cell.'
    )

print(f'Using dataset: {DATASET_ROOT}')
print((DATASET_ROOT / 'data.yaml').read_text())

In [ ]:
# Audit the export before spending GPU time. This validates the layout and reports class balance.
EXPECTED_NAMES = ['customer', 'staff']

def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

totals = Counter()
for split in ('train', 'valid', 'test'):
    images = image_files(DATASET_ROOT / split / 'images')
    labels = {p.stem: p for p in (DATASET_ROOT / split / 'labels').glob('*.txt')}
    image_stems = {p.stem for p in images}
    missing = image_stems - labels.keys()
    orphaned = labels.keys() - image_stems
    if missing or orphaned:
        raise ValueError(f'{split}: missing labels={len(missing)}, labels without images={len(orphaned)}')

    split_counts = Counter()
    for label_path in labels.values():
        for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
            values = line.split()
            if len(values) != 5:
                raise ValueError(f'Malformed label: {label_path}:{line_number}')
            class_id = int(values[0])
            if class_id not in (0, 1):
                raise ValueError(f'Unexpected class id {class_id}: {label_path}:{line_number}')
            split_counts[class_id] += 1

    totals.update(split_counts)
    print(f"{split:5} images={len(images):3}  customer={split_counts[0]:4}  staff={split_counts[1]:4}")

assert sum(totals.values()) > 0
print(f"Total boxes: {sum(totals.values())}; class names: {EXPECTED_NAMES}")

In [ ]:
# Write a Kaggle-safe data file with absolute paths.
# The export's relative ../train paths can resolve incorrectly after a Kaggle attachment.
KAGGLE_DATA_YAML = Path('/kaggle/working/staff_customer_kaggle.yaml')
kaggle_config = {
    'train': str(DATASET_ROOT / 'train' / 'images'),
    'val': str(DATASET_ROOT / 'valid' / 'images'),
    'test': str(DATASET_ROOT / 'test' / 'images'),
    'nc': 2,
    'names': EXPECTED_NAMES,
}
KAGGLE_DATA_YAML.write_text(yaml.safe_dump(kaggle_config, sort_keys=False))
print(KAGGLE_DATA_YAML.read_text())

## Train

`yolov8n` is a fast baseline appropriate for Kaggle. After the pipeline works, try `yolov8s` only if the GPU has enough memory. `batch=-1` asks Ultralytics to choose a safe batch size automatically.

In [ ]:
RUNS_ROOT = Path('/kaggle/working/staff_customer_runs')
RUN_NAME = 'yolov8n_640'

model = YOLO('yolov8n.pt')
train_metrics = model.train(
    data=str(KAGGLE_DATA_YAML),
    epochs=75,
    imgsz=640,
    batch=-1,
    device=DEVICE,
    workers=2,
    patience=20,
    optimizer='auto',
    cos_lr=True,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

TRAIN_DIR = RUNS_ROOT / RUN_NAME
BEST_WEIGHTS = TRAIN_DIR / 'weights' / 'best.pt'
assert BEST_WEIGHTS.exists(), f'No best weights found at {BEST_WEIGHTS}'
print(f'Best checkpoint: {BEST_WEIGHTS}')

In [ ]:
# Evaluate once on the export's test split. Treat the values as optimistic because of temporal leakage.
# Unlike model.train(), Ultralytics 8.4 does not accept batch=-1 (auto-batch) in model.val().
EVAL_BATCH = 16
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=str(KAGGLE_DATA_YAML),
    split='test',
    imgsz=640,
    batch=EVAL_BATCH,
    device=DEVICE,
    conf=0.001,
    plots=True,
    project=str(RUNS_ROOT),
    name='test_evaluation',
    exist_ok=True,
)

print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall:    {test_metrics.box.mr:.4f}")
print(f"mAP@50:    {test_metrics.box.map50:.4f}")
print(f"mAP@50-95: {test_metrics.box.map:.4f}")
print('Do not interpret these as production accuracy; evaluate later on unseen stores/cameras/sequences.')

In [ ]:
# Visual sanity check: predictions on eight test images.
from IPython.display import display, Image

test_images = image_files(DATASET_ROOT / 'test' / 'images')
sample_images = test_images[:8]
PREDICT_NAME = 'test_samples'
best_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(RUNS_ROOT),
    name=PREDICT_NAME,
    exist_ok=True,
)

prediction_dir = RUNS_ROOT / PREDICT_NAME
for prediction in sorted(prediction_dir.glob('*')):
    display(Image(filename=str(prediction), width=500))

In [ ]:
# Make the trained weights, plots, confusion matrix, and metrics available in the Kaggle Output tab.
import shutil

archive_base = '/kaggle/working/staff_customer_yolov8_results'
archive_path = shutil.make_archive(archive_base, 'zip', root_dir=RUNS_ROOT)
print(f'Weights: {BEST_WEIGHTS}')
print(f'Results archive: {archive_path}')
print('Use Save Version in Kaggle, then download best.pt or the results ZIP from the Output tab.')

## Responsible next step

For our application, consume a staff prediction only as weak visual evidence. Keep people as `UNKNOWN` unless an employee is manager-registered or a customer satisfies the configured entry/context rule. Before relying on a role model, rebuild the dataset using distinct recordings, cameras, and stores in separate partitions, and measure performance on footage the model has never encountered.